# 🛸 Unsupervised Drone Telemetry Exploration & Kinematic Distributions

This notebook performs Exploratory Data Analysis (EDA) on genuine DJI flight telemetry, hardware spoofer broadcasts (ESP32), and programmatic spoofing trajectories.

## 1. Setup & Environment

In [ ]:
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path("..").resolve()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Operating Directory:", os.getcwd())
from implement.utils.helper import get_or_preprocess_dji_dataset, get_or_preprocess_esp32_dataset, UNSUPERVISED_FEATURES

## 2. Load Datasets

In [ ]:
dji_df = get_or_preprocess_dji_dataset(features=UNSUPERVISED_FEATURES)
esp32_df = get_or_preprocess_esp32_dataset()

print(f"Normal DJI Samples: {len(dji_df)} across {dji_df["flight_id"].nunique()} flights")
print(f"Spoofed / Attack Samples: {len(esp32_df)} across {esp32_df["flight_id"].nunique()} flights")

## 3. Kinematic Feature Distributions (KDE Density)

In [ ]:
features_to_plot = ["ground_speed", "vertical_speed", "acceleration", "turn_rate", "path_curvature", "prediction_error"]
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for idx, feat in enumerate(features_to_plot):
    if feat in dji_df.columns and feat in esp32_df.columns:
        sns.kdeplot(dji_df[feat].dropna(), ax=axes[idx], label="Normal DJI", color="#1f77b4", fill=True, alpha=0.3)
        sns.kdeplot(esp32_df[feat].dropna(), ax=axes[idx], label="Spoofed / Attack", color="#d62728", fill=True, alpha=0.3)
        axes[idx].set_title(f"Distribution of {feat}")
        axes[idx].legend()

plt.tight_layout()
plt.show()

## 4. Feature Correlation Matrix (Normal vs Spoofed)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

corr_dji = dji_df[UNSUPERVISED_FEATURES].corr()
corr_spoof = esp32_df[[f for f in UNSUPERVISED_FEATURES if f in esp32_df.columns]].corr()

sns.heatmap(corr_dji, ax=axes[0], annot=True, fmt=".2f", cmap="coolwarm", cbar=False)
axes[0].set_title("Normal DJI Feature Correlation Matrix")

sns.heatmap(corr_spoof, ax=axes[1], annot=True, fmt=".2f", cmap="coolwarm")
axes[1].set_title("Spoofed Telemetry Feature Correlation Matrix")

plt.tight_layout()
plt.show()